# 🫀 rPPG-Toolbox: DeepPhys Training & Evaluation on MMPD Dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drozanes/rPPG-Toolbox/blob/main/MMPD_DeepPhys_Colab.ipynb)

This notebook runs **DeepPhys** training and evaluation on the full **MMPD** dataset (all 33 subjects, 660 videos) using an NVIDIA T4 GPU on Google Colab.

### Key Features:
- **Zero Disk Preprocessing Footprint:** Uses Online Memory Streaming (MMPDLoader with worker-level LRU caching) to bypass the ~940 GB disk space requirement.
- **Fast Training:** Optimized for CUDA with contiguous tensor layout and OnlineVideoSampler.
- **Validation Loss Tracking & Early Stopping:** Evaluates validation loss at each epoch and automatically stops when the model converges or begins overfitting.
- **Automatic Model Selection:** Evaluates the best checkpoint on the test set and computes MAE, RMSE, MAPE, Pearson r, and SNR.


In [ ]:
# 1. Verify GPU Allocation (Ensure you are on GPU runtime: Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

In [ ]:
# 2. Clone the rPPG-Toolbox Repository and Install Dependencies
!git clone https://github.com/drozanes/rPPG-Toolbox.git
%cd rPPG-Toolbox

!pip install -q pyyaml scipy opencv-python tqdm pandas matplotlib scikit-learn huggingface_hub wandb

In [ ]:
# 3. Download the Raw MMPD Mini Dataset from Hugging Face (~44.7 GB)
# Takes approx 12-15 minutes on Colab's high-speed connection
import os
from huggingface_hub import snapshot_download

data_dir = "./MMPD_dataset"
os.makedirs(data_dir, exist_ok=True)

print("Downloading MMPD dataset from Hugging Face...")
snapshot_download(
    repo_id="Sentio/MMPD",
    repo_type="dataset",
    local_dir=data_dir,
    local_dir_use_symlinks=False,
    resume_download=True
)

import glob
mat_files = glob.glob(f"{data_dir}/**/*.mat", recursive=True)
print(f"Successfully downloaded {len(mat_files)} / 660 .mat files!")
assert len(mat_files) == 660, f"Expected 660 files, found {len(mat_files)}"

In [ ]:
# 4. Launch DeepPhys Online Training with Validation Tracking & Early Stopping
# Training split: Subjects 1-26 (5,200 chunks)
# Validation split: Subjects 27-29 (600 chunks)
# Test split: Subjects 30-33 (800 chunks)
# Speed on Tesla T4: ~6.5 minutes per epoch

!python main.py --config_file configs/train_configs/MMPD_DEEPPHYS_ONLINE.yaml --device cuda --num_workers 2

In [ ]:
# 5. Display Benchmark Results
import pandas as pd
if os.path.exists("rPPG_results_database.csv"):
    df = pd.read_csv("rPPG_results_database.csv")
    print("\n=== Benchmark Database (MMPD Entries) ===")
    display(df[df['Dataset'] == 'MMPD'])
else:
    print("Results database not found.")

In [ ]:
# 6. Bundle & Download Trained Weights & Results
!zip -q -r mmpd_deepphys_trained_model.zip runs/
from google.colab import files
files.download("mmpd_deepphys_trained_model.zip")
print("Downloaded model checkpoint and evaluation outputs!")